# Create Arrow files from fragments

Optional starting point: run this notebook, then `02_analysis.ipynb`. Start Jupyter in this folder using the restored R environment. Inputs are in `inputs/`; Arrow files are written directly to `arrow_files/`.

The analysis uses GRCm39/mm39, the Cell Ranger ARC 2024-A filtered GENCODE M33 annotation and the native excluded ranges documented in `reference/provenance.json`.


# Create Arrow File
This notebook creates an Arrow File from the fragment files for ArchR analysis.

In [ ]:
# Restore the environment once as described in README.md before running this notebook.


In [ ]:
version

In [ ]:
source(file.path(Sys.getenv("ARCHR_PROJECT_DIR", unset = getwd()), "setup.R"))


In [ ]:
library(ArchR)

Specify fragment file names

In [ ]:
inputFiles <- c(HFD = file.path(run_dir, "inputs/HFD.fragments.tsv.gz"),
                LFD = file.path(run_dir, "inputs/LFD.fragments.tsv.gz"))


## Cell Ranger called-cell barcodes

Retain the supplied barcodes that passed Cell Ranger cell calling, then apply the original Arrow thresholds below.


In [ ]:
hfd_filtered_barcodes <- readLines(file.path(run_dir, "inputs/HFD.barcodes.tsv"))
lfd_filtered_barcodes <- readLines(file.path(run_dir, "inputs/LFD.barcodes.tsv"))
length(hfd_filtered_barcodes)
length(lfd_filtered_barcodes)


In [ ]:
addArchRThreads(threads = 16)
# ArchR 1.0.3 requires explicit custom annotations for mm39.
stopifnot(all(genome(gene_anno_mm39$genes) == "mm39"))
validate_staged_inputs()
if (any(file.exists(file.path(arrow_dir, c("HFD.arrow", "LFD.arrow")))))
    stop("Move existing Arrows out of arrow_files/ before rebuilding from fragments.")


In [ ]:
# ArchR writes Arrow files into the working directory.
# Keep its temporary working path free of spaces, then copy into arrow_files/.
arrow_runtime <- tempfile("ArchR_arrows_", tmpdir = tempdir())
dir.create(arrow_runtime)
setwd(arrow_runtime)
# Create Arrow files with specified barcodes and very low thresholds (QC already done by only using the barcodes that passed the cellranger QC)
ArrowFiles <- createArrowFiles(
  inputFiles = unname(inputFiles[c("HFD", "LFD")]),
  sampleNames = c("HFD", "LFD"),
  validBarcodes = list(
    HFD = hfd_filtered_barcodes,
    LFD = lfd_filtered_barcodes
  ),
  geneAnnotation = gene_anno_mm39,
  genomeAnnotation = genome_anno_mm39,
  minTSS = 1,
  minFrags = 100,
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
  force = FALSE
)

In [ ]:
print(ArrowFiles)

In [ ]:
# Copy the completed Arrows into the location used by the analysis notebook.
for (sample in c("HFD", "LFD")) {
    from <- file.path(arrow_runtime, paste0(sample, ".arrow"))
    to <- file.path(arrow_dir, paste0(sample, ".arrow"))
    if (file.exists(to)) stop("Arrow already exists: ", to,
        ". Move existing Arrows elsewhere before using the fragment workflow.")
    if (!file.copy(from, to)) stop("Could not copy Arrow: ", sample)
}
ArrowFiles <- setNames(file.path(arrow_dir, c("HFD.arrow", "LFD.arrow")), c("HFD", "LFD"))
writeLines(capture.output(sessionInfo()), file.path(project_dir, "output", "logs", "arrow_sessionInfo.txt"))
setwd(project_dir)
ArrowFiles
